In [0]:
%pip install mlflow pydantic openpyxl

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/query-planner-demo")

Agent = WorkspaceClient()

PLANNER_PROMPT = """
You are a healthcare query planning agent for Indian medical facility search.

Split the user request into structured components.
Do not recommend any facility.
Do not invent facts.

Return valid JSON with exactly these keys:
medical_need
required_capabilities
preferred_specialties
must_have_features
soft_preferences
location_scope
urgency
distance_preference
trust_threshold
reasoning_steps

Schema rules:
- required_capabilities: array of strings
- preferred_specialties: array of strings
- must_have_features: array of strings
- soft_preferences: array of strings
- location_scope: object with keys state, city, district, rural_hint, pin_code, user_latitude, user_longitude
- urgency: one of [low, medium, high, unknown]
- distance_preference: object with keys nearest_only, max_distance_km
- trust_threshold: number between 0 and 1
- reasoning_steps:
If unknown, return null or [].
"""

@mlflow.trace
def plan_query(user_query: str):
    response = Agent.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=PLANNER_PROMPT),
            ChatMessage(role=ChatMessageRole.USER, content=user_query),
        ],
        temperature=0.1,
        max_tokens=900
    )
    return response.choices[0].message.content

user_query = "Find the nearest facility in rural Bihar that can perform an emergency appendectomy and typically leverages parttime doctors"
raw_plan = plan_query(user_query)
print(raw_plan)

In [0]:
import json
import re

def parse_json_safely(text: str):
    text = text.strip()
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError("No valid JSON found in model output")

query_plan = parse_json_safely(raw_plan)
query_plan

In [0]:
from pyspark.sql import functions as F

facilities_df = spark.table("facilities_scored")

facilities_search_df = (
    facilities_df
    .withColumn("specialties_text", F.lower(F.concat_ws(" ", F.col("specialties"))))
    .withColumn("procedures_text", F.lower(F.concat_ws(" ", F.col("procedures"))))
    .withColumn("equipment_text", F.lower(F.concat_ws(" ", F.col("equipment"))))
    .withColumn("capabilities_text", F.lower(F.concat_ws(" ", F.col("capabilities"))))
    .withColumn("trust_flags_text", F.lower(F.concat_ws(" ", F.col("trust_flags"))))
    .withColumn("raw_notes_text", F.lower(F.coalesce(F.col("raw_notes"), F.lit(""))))
    .withColumn("reasoning_text", F.lower(F.coalesce(F.col("reasoning"), F.lit(""))))
    .withColumn("trust_score_safe", F.coalesce(F.col("trust_score"), F.lit(0.0)))
)